# 🟡 Solution: Winding Number

**Primitive:** broadcasting cross-product winding count

**Reduction:** Sum upward crossings (left of edge) minus downward crossings (right of edge) over all V edges, broadcast over `(P,1)` vs `(1,V)`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitive: broadcasting winding count (cross-product sign)

import numpy as np

def winding_number(points, polygon):
    points  = np.asarray(points,  dtype=float)
    polygon = np.asarray(polygon, dtype=float)
    px = points[:, 0:1]; py = points[:, 1:2]        # (P, 1)
    v1 = polygon; v2 = np.roll(polygon, -1, axis=0)  # (V, 2)
    x1, y1 = v1[:,0], v1[:,1]; x2, y2 = v2[:,0], v2[:,1]  # (V,)
    # cross product: (x2-x1)*(py-y1) - (y2-y1)*(px-x1)  → (P, V)
    cross = (x2-x1)*(py-y1) - (y2-y1)*(px-x1)
    up   = (y1 <= py) & (py < y2) & (cross > 0)   # upward crossing left
    down = (y2 <= py) & (py < y1) & (cross < 0)   # downward crossing right
    return (np.sum(up, axis=1) - np.sum(down, axis=1)).astype(int)

In [ ]:
# 🔍 Verify solution
# CCW square: inside → +1, outside → 0
ccw_sq = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])
pts = np.array([[0.5,0.5],[2.,0.5],[-0.5,0.5]])
print(winding_number(pts, ccw_sq))   # expect [1, 0, 0]

# CW square: inside → -1
cw_sq = ccw_sq[::-1].copy()
print(winding_number(pts, cw_sq))    # expect [-1, 0, 0]

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: CCW square ─────────────────────────────────────────────────────
ccw=np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])
pts=np.array([[0.5,0.5],[2.,0.5],[-0.5,0.5]])
r=winding_number(pts,ccw)
assert r[0]==1 and r[1]==0 and r[2]==0, f"CCW square: {r}"
print("Test 1 passed: CCW square (+1 inside, 0 outside)")

# ── Test 2: CW square ──────────────────────────────────────────────────────
cw=ccw[::-1].copy()
r2=winding_number(np.array([[0.5,0.5]]),cw)
assert r2[0]==-1, f"CW inside: {r2[0]}"
print("Test 2 passed: CW square (-1 inside)")

# ── Test 3: CCW and CW are negatives ──────────────────────────────────────
big=np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]])
big_cw=big[::-1].copy()
ip=np.array([[1.,1.]])
r3a=winding_number(ip,big); r3b=winding_number(ip,big_cw)
assert r3a[0]!=0 and r3b[0]!=0 and r3a[0]==-r3b[0], f"CCW={r3a[0]}, CW={r3b[0]}"
print("Test 3 passed: CCW and CW are negatives of each other")

# ── Test 4: batch 20 points vs circle polygon ──────────────────────────────
n=12; a=np.linspace(0,2*np.pi,n,endpoint=False)
poly=np.stack([np.cos(a),np.sin(a)],axis=1)
rng=np.random.default_rng(0); pts4=rng.uniform(-1.5,1.5,(20,2))
r4=winding_number(pts4,poly)
d=np.hypot(pts4[:,0],pts4[:,1])
assert (r4[d<0.85]!=0).all() and (r4[d>1.15]==0).all(), "Circle winding mismatch"
print("Test 4 passed: batch points vs circular polygon")

# ── Test 5: P=5000 ─────────────────────────────────────────────────────────
n=50; a=np.linspace(0,2*np.pi,n,endpoint=False)
poly5=np.stack([np.cos(a),np.sin(a)],axis=1)
pts5=rng.uniform(-1.5,1.5,(5000,2))
t0=time.time(); r5=winding_number(pts5,poly5); elapsed=time.time()-t0
assert r5.shape==(5000,) and elapsed<3.0, f"Shape/time: {r5.shape}, {elapsed:.2f}s"
print(f"Test 5 passed: P=5000 ({elapsed:.3f}s)")

print("\nAll tests passed!")